## Deep Learning for Mortality Prediction (DLMP)

### Import packages 

In [6]:
import tensorflow as tf
import numpy as np
import os as os
tfkl = tf.keras.layers
import random

### Import functions

In [7]:
import training_functions
import importlib

importlib.reload(training_functions)

<module 'training_functions' from '/Users/paigepark/Desktop/repos/deep-mort/code_expanding_window/training_functions.py'>

### Import data

#### Country data

In [8]:
splits = [
    (1959, 1979, 1980, 1989),
    (1959, 1989, 1990, 1999),
    (1959, 1999, 2000, 2009),
    (1959, 2009, 2010, 2019),
]

In [9]:
geos_key = np.load('../data/geos_key.npy')

In [10]:
geo_dict = {int(code): geo for geo, code in geos_key}

In [13]:
unique_vals = tf.unique(country_training[:, 0]).y
country_geo_dim = np.array(tf.size(unique_vals)).item() + 50
country_geo_dim

87

#### All Country Model

These models are those used in the paper to produce all of main figures/table in the paper.

In [15]:
# run all country model across expanding-window splits
forecast_horizon = 10
for train_start, train_end, test_start, test_end in splits:
    split_label = f"train{train_start}-{train_end}_test{test_start}-{test_end}"
    print(f"\n=== Split: {split_label} ===")

    country_training = np.loadtxt(f'../data/country_training_{train_start}_{train_end}.txt')
    country_test = np.loadtxt(f'../data/country_test_{test_start}_{test_end}.txt')

    min_year = float(country_training[:, 2].min())
    max_train_year = float(country_training[:, 2].max())
    year_denom = max_train_year + forecast_horizon - min_year

    country_train_prepped = training_functions.prep_data(
        country_training, mode="train",
        min_year=min_year, max_train_year=max_train_year,
        forecast_horizon=forecast_horizon, changeratetolog=True,
    )
    country_test_prepped = training_functions.prep_data(
        country_test, mode="test",
        min_year=min_year, max_train_year=max_train_year,
        forecast_horizon=forecast_horizon, changeratetolog=True,
    )

    unique_vals = tf.unique(country_training[:, 0]).y
    country_geo_dim = np.array(tf.size(unique_vals)).item() + 53

    for i in range(1, 4):
        np.random.seed(i)
        tf.random.set_seed(i)
        random.seed(i)
        os.environ['PYTHONHASHSEED'] = str(i)

        model_country, loss_info_country = training_functions.run_deep_model(
            country_train_prepped, country_test_prepped, country_geo_dim,
            epochs=20,
            steps_per_epoch=1405,
            lograte=True,
        )

        training_input_features = (
            tf.convert_to_tensor((country_training[:, 2] - min_year) / year_denom, dtype=tf.float32),
            tf.convert_to_tensor(country_training[:, 3], dtype=tf.float32),
            tf.convert_to_tensor(country_training[:, 0], dtype=tf.float32),
            tf.convert_to_tensor(country_training[:, 1], dtype=tf.float32),
        )

        test_input_features = (
            tf.convert_to_tensor((country_test[:, 2] - min_year) / year_denom, dtype=tf.float32),
            tf.convert_to_tensor(country_test[:, 3], dtype=tf.float32),
            tf.convert_to_tensor(country_test[:, 0], dtype=tf.float32),
            tf.convert_to_tensor(country_test[:, 1], dtype=tf.float32),
        )

        training_predictions = model_country.predict(training_input_features)
        test_predictions = model_country.predict(test_input_features)

        inputs = np.delete(country_training, 4, axis=1)
        training_predictions = np.column_stack((inputs, training_predictions))
        inputs_test = np.delete(country_test, 4, axis=1)
        test_predictions = np.column_stack((inputs_test, test_predictions))

        model_country.save(f"../models/dl_country_model_{split_label}_seed{i}.keras")
        np.savetxt(f"../data/dl_country_fitted_{split_label}_seed{i}.txt", training_predictions)
        np.savetxt(f"../data/dl_country_forecast_{split_label}_seed{i}.txt", test_predictions)

        print(f"  Split {split_label} | Iteration {i} complete")


=== Split: train1959-1979_test1980-1989 ===
Epoch 1/20
1405/1405 - 6s - 4ms/step - loss: 1.8378 - val_loss: 0.1656 - learning_rate: 1.0000e-03
Epoch 2/20
1405/1405 - 5s - 3ms/step - loss: 0.2997 - val_loss: 0.1456 - learning_rate: 1.0000e-03
Epoch 3/20
1405/1405 - 5s - 3ms/step - loss: 0.2253 - val_loss: 0.1599 - learning_rate: 1.0000e-03
Epoch 4/20
1405/1405 - 5s - 3ms/step - loss: 0.1879 - val_loss: 0.1723 - learning_rate: 1.0000e-03
Epoch 5/20
1405/1405 - 5s - 3ms/step - loss: 0.1730 - val_loss: 0.1413 - learning_rate: 1.0000e-03
Epoch 6/20
1405/1405 - 5s - 3ms/step - loss: 0.1643 - val_loss: 0.1594 - learning_rate: 1.0000e-03
Epoch 7/20
1405/1405 - 5s - 3ms/step - loss: 0.1602 - val_loss: 0.1293 - learning_rate: 1.0000e-03
Epoch 8/20
1405/1405 - 5s - 3ms/step - loss: 0.1537 - val_loss: 0.1389 - learning_rate: 1.0000e-03
Epoch 9/20
1405/1405 - 5s - 4ms/step - loss: 0.1523 - val_loss: 0.1407 - learning_rate: 1.0000e-03
Epoch 10/20
1405/1405 - 5s - 4ms/step - loss: 0.1464 - val_loss: